# Municipal risk

The municipal risk profile ranks municipalities into risk levels from their emergency history and from the shallow earthquakes near them. Level 1 is the lowest risk. Municipalities are grouped by Ward hierarchical clustering.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT))
DATA = ROOT / "data"

## Seismic features

`seismic_features` counts, for every municipality, the shallow earthquakes of magnitude 4 or more within 50 km of its centroid, and records the largest one.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
from scipy.cluster.hierarchy import dendrogram, linkage
from sklearn.preprocessing import StandardScaler

from ml.data import load_frame, municipality_centroids, read_divipola
from ml.municipal_risk import municipal_features, train_municipal_risk
from ml.seismic import read_catalog, seismic_features

frame = load_frame(DATA / "ungrd_emergencies_v2.csv", DATA / "divipola_dane.csv")
divipola = read_divipola(DATA / "divipola_dane.csv")
seismic = seismic_features(municipality_centroids(divipola), read_catalog(DATA / "sgc_seismicity_m25.csv"))
near_events = sum(entry.shallow_event_count > 0 for entry in seismic.values())
print(f"Municipalities with at least one nearby shallow event: {near_events}")

## Train the risk model

`train_municipal_risk` clusters the municipalities with at least three emergencies and keeps the number of levels with the best silhouette score.

In [ ]:
result = train_municipal_risk(frame, seismic)
model = result.model
metrics = result.metrics
print(f"Municipalities: {metrics['municipalities']}")
print(f"Levels (k): {metrics['k']}")
print(f"Silhouette: {metrics['silhouette']:.3f}")
print(f"Stability (adjusted Rand index): {metrics['stability_ari']:.3f}")

## Silhouette by number of levels

The silhouette score for each number of levels tried. Higher is better.

In [ ]:
by_k = pd.Series(metrics["silhouette_by_k"])
ax = by_k.plot(marker="o", figsize=(6, 3.5))
ax.set_xlabel("Number of levels (k)")
ax.set_ylabel("Silhouette")
plt.show()

## Dendrogram

The tree of merges behind the clustering, on the standardized municipal features. It is cut to the last 30 merged groups to stay readable.

In [ ]:
features = municipal_features(frame, seismic, model.type_names)
tree = linkage(StandardScaler().fit_transform(features), "ward")
fig, ax = plt.subplots(figsize=(12, 4))
dendrogram(tree, truncate_mode="lastp", p=30, ax=ax)
ax.set_xlabel("Merged groups (municipalities per group)")
ax.set_ylabel("Ward distance")
plt.show()

## Level sizes

The number of municipalities in each risk level.

In [ ]:
print(pd.Series(metrics["level_sizes"], name="municipalities").rename_axis("level").to_string())

## Highest-risk municipalities

The 15 municipalities with the highest level, ordered by level and then by severity score. Names come from the DIVIPOLA table.

In [ ]:
names = dict(zip(divipola["municipality_code"], divipola["municipality_key"].str.title()))
ranking = pd.DataFrame(
    {
        "code": list(model.levels),
        "municipality": [names.get(code, code) for code in model.levels],
        "level": list(model.levels.values()),
        "score": [round(model.scores[code], 2) for code in model.levels],
    }
)
print(ranking.sort_values(["level", "score"], ascending=False).head(15).to_string(index=False))

## Santander

How the municipalities of Santander (department code 68) spread across the levels, and the level of Los Santos (68418).

In [ ]:
santander = pd.Series({code: level for code, level in model.levels.items() if code.startswith("68")})
print(santander.value_counts().sort_index().rename_axis("level").to_string())
print(f"Los Santos (68418): level {model.level_of('68418')}")